# HuskySwap: serve the trained student (Qwen3-4B + your LoRA adapters)

**Runtime → Change runtime type → L4**, then **Runtime → Run all**. Approve Google Drive access when asked.

Serves `Qwen/Qwen3-4B-Instruct-2507` with every adapter listed in `RUNS` (from `MyDrive/huskyswap/adapters/`), side by side, so different training runs can be compared without restarting. Opens a Cloudflare tunnel, checks each adapter is really being served, and prints 4 lines for `.env.local`.

Keep this tab open while the eval runs; stop the runtime when done.

In [ ]:
# 1. Settings, Google Drive, and a check that every adapter exists
RUNS = ["stage2-r2", "stage2-r1"]   # training runs to serve side by side; the first is the default
ADAPTERS = {run: f"/content/drive/MyDrive/huskyswap/adapters/{run}" for run in RUNS}
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
PORT = 8001

import os, json
from google.colab import drive
drive.mount("/content/drive")
for run, path in ADAPTERS.items():
    assert os.path.exists(f"{path}/adapter_config.json"), f"No adapter at {path}: check the folder name in Drive"
    print(f"{run}: rank {json.load(open(f'{path}/adapter_config.json')).get('r')}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Stop any earlier server or tunnel in this session, then install vLLM if needed
!pkill -f "vllm serve" || true
!pkill -f cloudflared || true
import importlib.util
if importlib.util.find_spec("vllm") is None:
    !pip install -q --upgrade vllm
!python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 3. Start the server with the adapter (a few minutes the first time)
import secrets, subprocess, time, urllib.request

API_KEY = secrets.token_urlsafe(24)
log = open("/content/vllm-trained.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", BASE_MODEL,
     "--host", "127.0.0.1", "--port", str(PORT),
     "--api-key", API_KEY,
     "--max-model-len", "32768",
     "--gpu-memory-utilization", "0.90",
     "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
     "--enable-lora", "--max-lora-rank", "64", "--max-loras", str(len(RUNS)),
     "--lora-modules", *[f"huskyswap-student-{run}={path}" for run, path in ADAPTERS.items()]],
    stdout=log, stderr=subprocess.STDOUT)

start = time.time()
while True:
    if server.poll() is not None:
        raise RuntimeError("vLLM exited; last log lines:\n" + open("/content/vllm-trained.log").read()[-3000:])
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5)
        break
    except Exception:
        time.sleep(10)
        print(f"loading... {int(time.time() - start)} s", end="\r")
print(f"\nserver ready after {int(time.time() - start)} s")

In [ ]:
# 4. Open the tunnel and wait until its address works
import re, socket
if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel_log = open("/content/tunnel-trained.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"], stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel-trained.log").read())
    if found:
        url = found.group(0)
        break
if not url:
    raise RuntimeError("No tunnel address; read /content/tunnel-trained.log")
for _ in range(45):  # a new address takes a few seconds to a minute to appear in DNS
    try:
        socket.gethostbyname(url.removeprefix("https://"))
        break
    except socket.gaierror:
        time.sleep(2)
print("tunnel:", url)

In [ ]:
# 5. Check the adapter is being served, test a tool call, print the settings
import urllib.error
headers = {"content-type": "application/json", "authorization": f"Bearer {API_KEY}"}

def get(path, body=None):
    req = urllib.request.Request(f"{url}{path}", data=json.dumps(body).encode() if body else None, headers=headers)
    for _ in range(12):
        try:
            return json.load(urllib.request.urlopen(req, timeout=120))
        except urllib.error.URLError as e:
            print(f"not reachable yet ({e.reason}); retrying...")
            time.sleep(5)
    raise RuntimeError("tunnel unreachable; read /content/tunnel-trained.log")

served = [m["id"] for m in get("/v1/models")["data"]]
print("served models:", served)
missing = [r for r in RUNS if f"huskyswap-student-{r}" not in served]
assert not missing, f"Not served: {missing}; read /content/vllm-trained.log"
SERVED_NAME = f"huskyswap-student-{RUNS[0]}"

t0 = time.time()
reply = get("/v1/chat/completions", {
    "model": SERVED_NAME,
    "messages": [{"role": "user", "content": "How many listings are there? Use the tool."}],
    "tools": [{"type": "function", "function": {"name": "run_sql", "description": "Run a SQL query",
               "parameters": {"type": "object", "properties": {"sql": {"type": "string"}}, "required": ["sql"]}}}],
    "max_tokens": 256})
print(f"test call: {time.time() - t0:.2f} s | {json.dumps(reply['choices'][0]['message'].get('tool_calls'))[:200]}")

print("\n=== Paste these 4 lines to Claude (or into .env.local) ===\n")
print(f"STUDENT_BASE_URL={url}/v1")
print(f"STUDENT_API_KEY={API_KEY}")
print(f"STUDENT_MODEL={SERVED_NAME}")
print(f"STUDENT_VERSION={RUNS[0]}")
print("Also served (pick with npm run eval -- --model=student --student-model=NAME):", ", ".join(f"huskyswap-student-{r}" for r in RUNS[1:]))